In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
import zipfile
import io
from pathlib import Path

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 1995
END_YEAR = 2023

print("Setup complete")

Setup complete


In [2]:
NE_URL = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"

world = gpd.read_file(NE_URL)

print(world.shape)
print(world.columns.tolist())
world.head(3)

(177, 169)
['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF', 'GEOUNIT', 'GU_A3', 'SU_DIF', 'SUBUNIT', 'SU_A3', 'BRK_DIFF', 'NAME', 'NAME_LONG', 'BRK_A3', 'BRK_NAME', 'BRK_GROUP', 'ABBREV', 'POSTAL', 'FORMAL_EN', 'FORMAL_FR', 'NAME_CIAWF', 'NOTE_ADM0', 'NOTE_BRK', 'NAME_SORT', 'NAME_ALT', 'MAPCOLOR7', 'MAPCOLOR8', 'MAPCOLOR9', 'MAPCOLOR13', 'POP_EST', 'POP_RANK', 'POP_YEAR', 'GDP_MD', 'GDP_YEAR', 'ECONOMY', 'INCOME_GRP', 'FIPS_10', 'ISO_A2', 'ISO_A2_EH', 'ISO_A3', 'ISO_A3_EH', 'ISO_N3', 'ISO_N3_EH', 'UN_A3', 'WB_A2', 'WB_A3', 'WOE_ID', 'WOE_ID_EH', 'WOE_NOTE', 'ADM0_ISO', 'ADM0_DIFF', 'ADM0_TLC', 'ADM0_A3_US', 'ADM0_A3_FR', 'ADM0_A3_RU', 'ADM0_A3_ES', 'ADM0_A3_CN', 'ADM0_A3_TW', 'ADM0_A3_IN', 'ADM0_A3_NP', 'ADM0_A3_PK', 'ADM0_A3_DE', 'ADM0_A3_GB', 'ADM0_A3_BR', 'ADM0_A3_IL', 'ADM0_A3_PS', 'ADM0_A3_SA', 'ADM0_A3_EG', 'ADM0_A3_MA', 'ADM0_A3_PT', 'ADM0_A3_AR', 'ADM0_A3_JP', 'ADM0_A3_KO', 'ADM0_A3_VN', 'ADM0_

,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,Unrecognized,Unrecognized,Unrecognized,NaN,NaN,Unrecognized,NaN,NaN,NaN,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."


In [3]:
FAOSTAT_URL = "https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip"

resp = requests.get(FAOSTAT_URL, timeout=300)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    print(z.namelist())
    with z.open(z.namelist()[0]) as f:
        qcl = pd.read_csv(f, encoding="latin1", low_memory=False)

print(qcl.shape)
qcl.head()

['Production_Crops_Livestock_E_All_Data_(Normalized).csv', 'Production_Crops_Livestock_E_AreaCodes.csv', 'Production_Crops_Livestock_E_Elements.csv', 'Production_Crops_Livestock_E_Flags.csv', 'Production_Crops_Livestock_E_ItemCodes.csv']
(4209110, 14)


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
0,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1961,1961,ha,0.0,A,NaN
1,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1962,1962,ha,0.0,A,NaN
2,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1963,1963,ha,0.0,A,NaN
3,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1964,1964,ha,0.0,A,NaN
4,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1965,1965,ha,0.0,A,NaN


In [4]:
candidates = qcl['Item'].unique()
matches = [c for c in candidates if any(k in c.lower() for k in ['maize', 'wheat', 'rice', 'soy'])]
matches

['Maize (corn)',
 'Rice',
 'Soya beans',
 'Wheat',
 'Soya bean oil',
 'Green corn (maize)',
 'Oil of maize',
 'Buckwheat']

In [5]:
print(world.shape)
print(list(world.columns))
world.drop(columns='geometry').head(3)

(177, 169)
['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF', 'GEOUNIT', 'GU_A3', 'SU_DIF', 'SUBUNIT', 'SU_A3', 'BRK_DIFF', 'NAME', 'NAME_LONG', 'BRK_A3', 'BRK_NAME', 'BRK_GROUP', 'ABBREV', 'POSTAL', 'FORMAL_EN', 'FORMAL_FR', 'NAME_CIAWF', 'NOTE_ADM0', 'NOTE_BRK', 'NAME_SORT', 'NAME_ALT', 'MAPCOLOR7', 'MAPCOLOR8', 'MAPCOLOR9', 'MAPCOLOR13', 'POP_EST', 'POP_RANK', 'POP_YEAR', 'GDP_MD', 'GDP_YEAR', 'ECONOMY', 'INCOME_GRP', 'FIPS_10', 'ISO_A2', 'ISO_A2_EH', 'ISO_A3', 'ISO_A3_EH', 'ISO_N3', 'ISO_N3_EH', 'UN_A3', 'WB_A2', 'WB_A3', 'WOE_ID', 'WOE_ID_EH', 'WOE_NOTE', 'ADM0_ISO', 'ADM0_DIFF', 'ADM0_TLC', 'ADM0_A3_US', 'ADM0_A3_FR', 'ADM0_A3_RU', 'ADM0_A3_ES', 'ADM0_A3_CN', 'ADM0_A3_TW', 'ADM0_A3_IN', 'ADM0_A3_NP', 'ADM0_A3_PK', 'ADM0_A3_DE', 'ADM0_A3_GB', 'ADM0_A3_BR', 'ADM0_A3_IL', 'ADM0_A3_PS', 'ADM0_A3_SA', 'ADM0_A3_EG', 'ADM0_A3_MA', 'ADM0_A3_PT', 'ADM0_A3_AR', 'ADM0_A3_JP', 'ADM0_A3_KO', 'ADM0_A3_VN', 'ADM0_

,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_VN,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,NaN,Unrecognized,Unrecognized,Unrecognized,NaN,NaN,Unrecognized,NaN,NaN,NaN


In [6]:
TARGET_CROPS = ["Maize (corn)", "Rice", "Soya beans", "Wheat"]

crops_df = qcl[qcl['Item'].isin(TARGET_CROPS)].copy()

print(crops_df.shape)
print(crops_df['Element'].unique())
print(crops_df['Unit'].unique())
crops_df.head()

(114876, 14)
<ArrowStringArray>
['Area harvested', 'Yield', 'Production']
Length: 3, dtype: str
<ArrowStringArray>
['ha', 'kg/ha', 't']
Length: 3, dtype: str


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
3818,2,'004,Afghanistan,56,'0112,Maize (corn),5312,Area harvested,1961,1961,ha,500000.0,A,NaN
3819,2,'004,Afghanistan,56,'0112,Maize (corn),5312,Area harvested,1962,1962,ha,500000.0,A,NaN
3820,2,'004,Afghanistan,56,'0112,Maize (corn),5312,Area harvested,1963,1963,ha,500000.0,A,NaN
3821,2,'004,Afghanistan,56,'0112,Maize (corn),5312,Area harvested,1964,1964,ha,505000.0,A,NaN
3822,2,'004,Afghanistan,56,'0112,Maize (corn),5312,Area harvested,1965,1965,ha,500000.0,A,NaN


In [7]:
yield_df = crops_df[
    (crops_df['Element'] == 'Yield') &
    (crops_df['Year'] >= START_YEAR) &
    (crops_df['Year'] <= END_YEAR)
].copy()

yield_df = yield_df[['Area Code', 'Area', 'Item', 'Year', 'Value', 'Unit']].rename(columns={
    'Area': 'country_name',
    'Item': 'crop',
    'Value': 'yield_kg_ha'
})

print(yield_df.shape)
yield_df.to_csv(RAW_DIR / "crop_yields_raw.csv", index=False)
yield_df.head()

(18039, 6)


,Area Code,country_name,crop,Year,yield_kg_ha,Unit
3916,2,Afghanistan,Maize (corn),1995,1656.3,kg/ha
3917,2,Afghanistan,Maize (corn),1996,1636.4,kg/ha
3918,2,Afghanistan,Maize (corn),1997,1500.0,kg/ha
3919,2,Afghanistan,Maize (corn),1998,1650.0,kg/ha
3920,2,Afghanistan,Maize (corn),1999,1500.0,kg/ha


In [8]:
relevant_cols = [c for c in world.columns if any(k in c.upper() for k in ['NAME', 'ISO_A3', 'ADMIN', 'CONTINENT', 'SOVEREIGNT'])]
print(relevant_cols)
world[relevant_cols].head(3)

['SOVEREIGNT', 'ADMIN', 'NAME', 'NAME_LONG', 'BRK_NAME', 'NAME_CIAWF', 'NAME_SORT', 'NAME_ALT', 'ISO_A3', 'ISO_A3_EH', 'CONTINENT', 'NAME_LEN', 'NAME_AR', 'NAME_BN', 'NAME_DE', 'NAME_EN', 'NAME_ES', 'NAME_FA', 'NAME_FR', 'NAME_EL', 'NAME_HE', 'NAME_HI', 'NAME_HU', 'NAME_ID', 'NAME_IT', 'NAME_JA', 'NAME_KO', 'NAME_NL', 'NAME_PL', 'NAME_PT', 'NAME_RU', 'NAME_SV', 'NAME_TR', 'NAME_UK', 'NAME_UR', 'NAME_VI', 'NAME_ZH', 'NAME_ZHT']


,SOVEREIGNT,ADMIN,NAME,NAME_LONG,BRK_NAME,NAME_CIAWF,NAME_SORT,NAME_ALT,ISO_A3,ISO_A3_EH,...,NAME_PL,NAME_PT,NAME_RU,NAME_SV,NAME_TR,NAME_UK,NAME_UR,NAME_VI,NAME_ZH,NAME_ZHT
0,Fiji,Fiji,Fiji,Fiji,Fiji,Fiji,Fiji,NaN,FJI,FJI,...,Fidżi,Fiji,Фиджи,Fiji,Fiji,Фіджі,فجی,Fiji,斐济,斐濟
1,United Republic of Tanzania,United Republic of Tanzania,Tanzania,Tanzania,Tanzania,Tanzania,Tanzania,NaN,TZA,TZA,...,Tanzania,Tanzânia,Танзания,Tanzania,Tanzanya,Танзанія,تنزانیہ,Tanzania,坦桑尼亚,坦尚尼亞
2,Western Sahara,Western Sahara,W. Sahara,Western Sahara,W. Sahara,Western Sahara,Western Sahara,NaN,ESH,ESH,...,Sahara Zachodnia,Sara Ocidental,Западная Сахара,Västsahara,Batı Sahra,Західна Сахара,مغربی صحارا,Tây Sahara,西撒哈拉,西撒哈拉


In [9]:
cols_to_check = ['NAME', 'ISO_A3', 'ISO_A3_EH', 'CONTINENT']
print(world[cols_to_check].head(5).to_string())
print()
print("Any '-99' in ISO_A3?:", (world['ISO_A3'] == '-99').sum())
print("Any '-99' in ISO_A3_EH?:", (world['ISO_A3_EH'] == '-99').sum())

                       NAME ISO_A3 ISO_A3_EH      CONTINENT
0                      Fiji    FJI       FJI        Oceania
1                  Tanzania    TZA       TZA         Africa
2                 W. Sahara    ESH       ESH         Africa
3                    Canada    CAN       CAN  North America
4  United States of America    USA       USA  North America

Any '-99' in ISO_A3?: 5
Any '-99' in ISO_A3_EH?: 3


In [10]:
print(world[world['ISO_A3_EH'] == '-99'][['NAME', 'SOVEREIGNT', 'ISO_A3', 'ISO_A3_EH']].to_string())

           NAME       SOVEREIGNT ISO_A3 ISO_A3_EH
160   N. Cyprus  Northern Cyprus    -99       -99
167  Somaliland       Somaliland    -99       -99
174      Kosovo           Kosovo    -99       -99


In [11]:
world_proj = world.to_crs("+proj=cea")
centroids = world_proj.centroid.to_crs(world.crs)

countries = world[['NAME', 'ISO_A3_EH', 'CONTINENT']].copy()
countries['lon'] = centroids.x
countries['lat'] = centroids.y
countries = countries.rename(columns={
    'NAME': 'country_name', 'ISO_A3_EH': 'iso3', 'CONTINENT': 'continent'
})

countries.to_csv(RAW_DIR / "countries_centroids.csv", index=False)
print(countries.shape)
print(countries.head(10).to_string())

(177, 5)
               country_name iso3      continent         lon        lat
0                      Fiji  FJI        Oceania  163.776407 -17.312775
1                  Tanzania  TZA         Africa   34.747669  -6.237731
2                 W. Sahara  ESH         Africa  -12.162184  24.249281
3                    Canada  CAN  North America  -97.644860  57.629665
4  United States of America  USA  North America -107.544729  42.075008
5                Kazakhstan  KAZ           Asia   67.254290  47.920842
6                Uzbekistan  UZB           Asia   63.284542  41.670262
7          Papua New Guinea  PNG        Oceania  145.315787  -6.442153
8                 Indonesia  IDN           Asia  117.415346  -2.209931
9                 Argentina  ARG  South America  -64.957299 -34.354498


In [12]:
import time

POWER_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"

test_row = countries.iloc[0]  # Fiji
print(test_row['country_name'], test_row['lon'], test_row['lat'])

params = {
    "parameters": "T2M,PRECTOTCORR",
    "community": "AG",
    "longitude": test_row['lon'],
    "latitude": test_row['lat'],
    "start": START_YEAR,
    "end": END_YEAR,
    "format": "JSON"
}

resp = requests.get(POWER_URL, params=params, timeout=30)
resp.raise_for_status()
data = resp.json()

t2m = data['properties']['parameter']['T2M']
precip = data['properties']['parameter']['PRECTOTCORR']

print("Number of keys:", len(t2m))
print("First 15 keys and values (T2M):")
for k in list(t2m.keys())[:15]:
    print(k, "->", t2m[k])
print()
print("Last 5 keys (T2M):")
for k in list(t2m.keys())[-5:]:
    print(k, "->", t2m[k])

Fiji 163.77640725005324 -17.312775222771094
Number of keys: 377
First 15 keys and values (T2M):
199501 -> 26.6
199502 -> 27.48
199503 -> 27.87
199504 -> 27.37
199505 -> 25.68
199506 -> 24.23
199507 -> 23.64
199508 -> 23.56
199509 -> 23.56
199510 -> 25.54
199511 -> 26.35
199512 -> 27.15
199513 -> 25.74
199601 -> 27.87
199602 -> 28.12

Last 5 keys (T2M):
202309 -> 24.59
202310 -> 25.11
202311 -> 25.56
202312 -> 26.95
202313 -> 26.15


In [13]:
yield_df = crops_df[
    (crops_df['Element'] == 'Yield') &
    (crops_df['Year'] >= START_YEAR) &
    (crops_df['Year'] <= END_YEAR)
].copy()

yield_df = yield_df[['Area Code', 'Area', 'Item', 'Year', 'Value', 'Unit']].rename(columns={
    'Area': 'country_name',
    'Item': 'crop',
    'Value': 'yield_kg_ha'
})

print(yield_df.shape)
yield_df.to_csv(RAW_DIR / "crop_yields_raw.csv", index=False)
print(yield_df.head(10).to_string())

(18039, 6)
      Area Code country_name          crop  Year  yield_kg_ha   Unit
3916          2  Afghanistan  Maize (corn)  1995       1656.3  kg/ha
3917          2  Afghanistan  Maize (corn)  1996       1636.4  kg/ha
3918          2  Afghanistan  Maize (corn)  1997       1500.0  kg/ha
3919          2  Afghanistan  Maize (corn)  1998       1650.0  kg/ha
3920          2  Afghanistan  Maize (corn)  1999       1500.0  kg/ha
3921          2  Afghanistan  Maize (corn)  2000       1197.9  kg/ha
3922          2  Afghanistan  Maize (corn)  2001       2000.0  kg/ha
3923          2  Afghanistan  Maize (corn)  2002       2980.0  kg/ha
3924          2  Afghanistan  Maize (corn)  2003        840.0  kg/ha
3925          2  Afghanistan  Maize (corn)  2004       1600.0  kg/ha


In [14]:
import time

def days_in_year(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365

weather_records = []
failed_countries = []

for i, row in countries.iterrows():
    params = {
        "parameters": "T2M,PRECTOTCORR",
        "community": "AG",
        "longitude": row['lon'],
        "latitude": row['lat'],
        "start": START_YEAR,
        "end": END_YEAR,
        "format": "JSON"
    }
    try:
        resp = requests.get(POWER_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        t2m = data['properties']['parameter']['T2M']
        precip = data['properties']['parameter']['PRECTOTCORR']

        for key in t2m:
            if key.endswith('13'):
                year = int(key[:4])
                avg_daily_precip = precip[key]
                weather_records.append({
                    'iso3': row['iso3'],
                    'country_name': row['country_name'],
                    'year': year,
                    'avg_temp_c': t2m[key],
                    'total_precip_mm': avg_daily_precip * days_in_year(year)
                })
    except Exception as e:
        failed_countries.append((row['country_name'], str(e)))

    if i % 20 == 0:
        print(f"Processed {i+1}/{len(countries)} countries...")

    time.sleep(0.5)

weather_df = pd.DataFrame(weather_records)

print(weather_df.shape)
print("Failed countries:", failed_countries)
print("Impossible temps (<-90C):", (weather_df['avg_temp_c'] < -90).sum())
print("Negative precip:", (weather_df['total_precip_mm'] < 0).sum())

weather_df.to_csv(RAW_DIR / "weather_raw.csv", index=False)
print(weather_df.head(10).to_string())

Processed 1/177 countries...
Processed 21/177 countries...
Processed 41/177 countries...
Processed 61/177 countries...
Processed 81/177 countries...
Processed 101/177 countries...
Processed 121/177 countries...
Processed 141/177 countries...
Processed 161/177 countries...
(5133, 5)
Failed countries: []
Impossible temps (<-90C): 0
Negative precip: 0
  iso3 country_name  year  avg_temp_c  total_precip_mm
0  FJI         Fiji  1995       25.74           773.80
1  FJI         Fiji  1996       26.01          1420.08
2  FJI         Fiji  1997       25.56           839.50
3  FJI         Fiji  1998       26.54          1175.30
4  FJI         Fiji  1999       25.93          1916.25
5  FJI         Fiji  2000       25.77          1277.34
6  FJI         Fiji  2001       26.02          1069.45
7  FJI         Fiji  2002       25.81          1273.85
8  FJI         Fiji  2003       26.01           952.65
9  FJI         Fiji  2004       26.02           669.78


In [15]:
WB_URL = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL"

params = {
    "format": "json",
    "date": f"{START_YEAR}:{END_YEAR}",
    "per_page": 20000
}

resp = requests.get(WB_URL, params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

print("Metadata:", data[0])

records = data[1]
pop_df = pd.DataFrame(records)
pop_df = pop_df[['countryiso3code', 'date', 'value']].rename(columns={
    'countryiso3code': 'iso3',
    'date': 'year',
    'value': 'population'
})
pop_df['year'] = pop_df['year'].astype(int)

print(pop_df.shape)
print("Missing population values:", pop_df['population'].isna().sum())
pop_df.to_csv(RAW_DIR / "population_raw.csv", index=False)
print(pop_df.head(10).to_string())

Metadata: {'page': 1, 'pages': 1, 'per_page': 20000, 'total': 7685, 'sourceid': '2', 'lastupdated': '2026-07-13'}
(7685, 3)
Missing population values: 29
  iso3  year   population
0  AFE  2023  750491370.0
1  AFE  2022  731821393.0
2  AFE  2021  713090928.0
3  AFE  2020  694446100.0
4  AFE  2019  675950189.0
5  AFE  2018  657801085.0
6  AFE  2017  640058741.0
7  AFE  2016  623369401.0
8  AFE  2015  607123269.0
9  AFE  2014  590968990.0


In [16]:
calendar_df = pd.DataFrame({'year': range(START_YEAR, END_YEAR + 1)})
calendar_df['decade'] = (calendar_df['year'] // 10) * 10
calendar_df['period_label'] = calendar_df['decade'].astype(str) + 's'
calendar_df['is_forecast'] = False  # all historical for now — forecast years get appended in Phase 5

calendar_df.to_csv(RAW_DIR / "calendar_raw.csv", index=False)
print(calendar_df.shape)
print(calendar_df.to_string())

(29, 4)
    year  decade period_label  is_forecast
0   1995    1990        1990s        False
1   1996    1990        1990s        False
2   1997    1990        1990s        False
3   1998    1990        1990s        False
4   1999    1990        1990s        False
5   2000    2000        2000s        False
6   2001    2000        2000s        False
7   2002    2000        2000s        False
8   2003    2000        2000s        False
9   2004    2000        2000s        False
10  2005    2000        2000s        False
11  2006    2000        2000s        False
12  2007    2000        2000s        False
13  2008    2000        2000s        False
14  2009    2000        2000s        False
15  2010    2010        2010s        False
16  2011    2010        2010s        False
17  2012    2010        2010s        False
18  2013    2010        2010s        False
19  2014    2010        2010s        False
20  2015    2010        2010s        False
21  2016    2010        2010s        False
22 

In [17]:
CCKP_URL = "https://cckpapi.worldbank.org/cckp/v1/cmip6-x0.25_climatology_tas,pr_climatology_annual_1995-2014_median_historical_ensemble_all_mean/IND"

resp = requests.get(CCKP_URL, params={"_format": "json"}, timeout=30)
print(resp.status_code)
print(resp.text[:2000])

200
{"metadata":{"apiVersion":"v1","status":"success","messages":[]},"data":{"pr":{"IND":{"1995-07":1186.02}},"tas":{"IND":{"1995-07":24.42}}}}


In [18]:
BASELINE_START = 1981
BASELINE_END = 2010

climate_records = []
failed_baseline = []

for i, row in countries.iterrows():
    params = {
        "parameters": "T2M,PRECTOTCORR",
        "community": "AG",
        "longitude": row['lon'],
        "latitude": row['lat'],
        "start": BASELINE_START,
        "end": BASELINE_END,
        "format": "JSON"
    }
    try:
        resp = requests.get(POWER_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        t2m = data['properties']['parameter']['T2M']
        precip = data['properties']['parameter']['PRECTOTCORR']

        annual_temps = []
        annual_precips = []
        for key in t2m:
            if key.endswith('13'):
                year = int(key[:4])
                annual_temps.append(t2m[key])
                annual_precips.append(precip[key] * days_in_year(year))

        climate_records.append({
            'iso3': row['iso3'],
            'country_name': row['country_name'],
            'baseline_avg_temp_c': np.mean(annual_temps),
            'baseline_avg_precip_mm': np.mean(annual_precips),
            'baseline_start': BASELINE_START,
            'baseline_end': BASELINE_END
        })
    except Exception as e:
        failed_baseline.append((row['country_name'], str(e)))

    if i % 20 == 0:
        print(f"Processed {i+1}/{len(countries)} countries...")

    time.sleep(0.5)

climate_df = pd.DataFrame(climate_records)

print(climate_df.shape)
print("Failed:", failed_baseline)
print("Impossible temps (<-90C):", (climate_df['baseline_avg_temp_c'] < -90).sum())

climate_df.to_csv(RAW_DIR / "climate_raw.csv", index=False)
print(climate_df.head(10).to_string())

Processed 1/177 countries...
Processed 21/177 countries...
Processed 41/177 countries...
Processed 61/177 countries...
Processed 81/177 countries...
Processed 101/177 countries...
Processed 121/177 countries...
Processed 141/177 countries...
Processed 161/177 countries...
(177, 6)
Failed: []
Impossible temps (<-90C): 0
  iso3              country_name  baseline_avg_temp_c  baseline_avg_precip_mm  baseline_start  baseline_end
0  FJI                      Fiji            25.750667             1204.904000            1981          2010
1  TZA                  Tanzania            23.270333              702.834667            1981          2010
2  ESH                 W. Sahara            24.652000               46.132000            1981          2010
3  CAN                    Canada            -6.177000              631.702667            1981          2010
4  USA  United States of America             5.666000              283.046667            1981          2010
5  KAZ                Kazakhsta